# MaxRL: Maximum Likelihood Reinforcement Learning

This notebook implements the **MaxRL** algorithm from [Zanette et al.](https://zanette-labs.github.io/MaxRL/) using the same character-level name generation task as the `nanoRLHF.ipynb` notebook, for an apples-to-apples comparison with REINFORCE and PPO.

**Key idea:** REINFORCE normalizes by total samples $N$, while MaxRL normalizes by the number of *successful* samples $K$. This single-line change targets a harmonic mixture of pass@k objectives instead of just pass@1, upweighting harder inputs where the model rarely succeeds.

For reference, see:
- [MaxRL: A Unified View](https://zanette-labs.github.io/MaxRL/) — project page
- [nanoRLHF.ipynb](nanoRLHF.ipynb) — the companion notebook with REINFORCE, PPO, and ES

Before we get started, let's make sure you have the correct version of PyTorch installed. If you have an older version or want to start fresh, you can uninstall the current PyTorch, torchvision, and torchaudio packages using the following command.

In [ ]:
# !pip3 uninstall -y torch torchvision torchaudio

If you need to install PyTorch with CUDA support (for GPU acceleration), you can use the following command. Make sure your environment supports CUDA 11.8.

In [ ]:
# !pip3 install torch --index-url https://download.pytorch.org/whl/cu118

Let's import PyTorch and check the version to make sure everything is set up correctly.

In [ ]:
import torch as pt
pt.__version__

Now, let's set up the device for computation. We'll use the GPU if it's available, otherwise we'll fall back to the CPU. The `nvidia-smi` command will show you information about your GPU if one is present.

In [ ]:
device = "cuda" if pt.cuda.is_available() else "cpu"
!nvidia-smi
device

We'll use a dataset of names for our experiments. Let's fetch a list of names from GitHub using Python's `urllib` library. This will give us a simple, real-world dataset to work with.

In [ ]:
import urllib
with urllib.request.urlopen('https://raw.githubusercontent.com/karpathy/makemore/master/names.txt') as resp:
  src = resp.read().decode('utf-8')
src

Now, let's convert the downloaded text into a Python list of names. We'll take a quick look at the first few names to make sure everything looks good.

In [ ]:
names = src.splitlines()
names[:5]

Next, we'll extract the unique characters (tokens) used in the dataset. We'll also add special tokens for the start (`_`) and end (`.`) of a name. This will help us later when we encode and decode names.

In [ ]:
tokens = "_" + "." + "".join(sorted(set("".join(names))))
tokens

To work with our tokens, we'll create two dictionaries: `stoi` (string-to-index) and `itos` (index-to-string). These will let us easily convert between characters and their corresponding indices.

In [ ]:
stoi = {v:k for k,v in enumerate(tokens)}

In [ ]:
print(stoi)

In [ ]:
itos = {k:v for k,v in enumerate(tokens)}

In [ ]:
print(itos)

Let's define two simple functions: `enc` to encode a name as a list of indices, and `dec` to decode a list of indices back to a string. This will make it easy to switch between string and numeric representations.

In [ ]:
enc = lambda name: [stoi[s] for s in name]
dec = lambda chars: "".join(itos[i] for i in chars)

In [ ]:
dec(enc('_emma.'))

Now, let's create a function `stot` that converts a string into a one-hot encoded tensor. This will be useful for representing names in a format suitable for neural networks.

In [ ]:
stot = lambda name: pt.nn.functional.one_hot(pt.tensor(enc(name)), len(tokens))

To go back from a one-hot encoded tensor to a string, we'll define the `ttos` function. This will help us interpret the outputs of our model later.

In [ ]:
ttos = lambda x: dec(x.argmax(-1).tolist())

Define a constant `CTX_SZ` to represent a window size used by the model to predict the next character.

In [ ]:
CTX_SZ = 4
CTX_SZ

Create helper functions `name_to_obs` and `name_to_action` for converting names to observation/action tensors.

In [ ]:
name_to_obs = lambda name: stot(name).unfold(0, CTX_SZ, 1)[:-1, :, :].permute(0, 2, 1)

In [ ]:
name_to_action = lambda name: stot(name)[CTX_SZ:]

Now you are ready to create the data set to train your model.

- convert all the names in `names` to the observations
- use `cat` to concatenate all the sparse embeddings of the names to a data set tensor `X_data`

In [ ]:
X_data = [name_to_obs(CTX_SZ * "_" + name + ".") for name in names]
X_data = pt.cat(X_data)

In [ ]:
[ttos(x) for x in X_data[:10]], X_data.shape

Move the `X_data` tensor to your `device`

In [ ]:
X_data = X_data.to(device)

Next, create the `y_data` tensor using `name_to_action`

In [ ]:
y_data = pt.cat([name_to_action(CTX_SZ * "_" + name + ".") for name in names])

In [ ]:
ttos(y_data[:10]), y_data.shape

Move the `y_data` tensor to your device

In [ ]:
y_data = y_data.to(device)

The data set should be shuffled before use for training. Use the `randperm` function to shuffle both the `X_data` and the `y_data` tensors along the 0th dimension.

In [ ]:
pt.manual_seed(42)
idx = pt.randperm(len(y_data))
idx.shape

In [ ]:
X_data, y_data = X_data[idx], y_data[idx]
X_data.shape, y_data.shape

Since the data set is fairly large, let's use a 90%, 5%, 5% split for the training, validation, and test data sets respectively.

In [ ]:
val_idx, test_idx = int(len(X_data) * .9), int(len(X_data) * .95)
X_train, y_train = X_data[:val_idx], y_data[:val_idx]
X_val, y_val = X_data[val_idx:test_idx], y_data[val_idx:test_idx]
X_test, y_test = X_data[test_idx:], y_data[test_idx:]

X_train.shape, X_val.shape, X_test.shape

In [ ]:
class CharModel(pt.nn.Module):
  def __init__(self, tokens_sz, ctx_sz, emb_sz, head_sz, n_heads, device):
    super().__init__()

    self.tok_emb = pt.nn.Embedding(tokens_sz, emb_sz, device = device)
    self.pos_emb = pt.nn.Embedding(ctx_sz, emb_sz, device = device)
    self.pos_idx = pt.arange(ctx_sz, device = device)

    self.kw = pt.nn.Linear(emb_sz, head_sz, device = device, bias = False)
    self.qw = pt.nn.Linear(emb_sz, head_sz, device = device, bias = False)
    self.vw = pt.nn.Linear(emb_sz, head_sz, device = device, bias = False)
    self.mhsa = pt.nn.MultiheadAttention(head_sz, n_heads, batch_first = True, device = device)
    self.mhsa_ln = pt.nn.LayerNorm(head_sz, device = device)
    self.flatten = pt.nn.Flatten(1)
    self.relu = pt.nn.ReLU()
    self.mhsa_head = pt.nn.Linear(head_sz * ctx_sz, tokens_sz, device = device)


  def forward(self, x):
    x = self.tok_emb(x) + self.pos_emb(self.pos_idx)
    q, k, v = self.qw(x), self.kw(x), self.vw(x)
    x, _ = self.mhsa(q, k, v,  )
    x = self.mhsa_ln(x)
    x = self.flatten(x)
    x = self.relu(x)
    x = self.mhsa_head(x)
    return x

In [ ]:
EMB_SZ = 32
MHSA_HEAD_SZ = 64
N_HEADS = 2
nn = CharModel(len(tokens), CTX_SZ, EMB_SZ, MHSA_HEAD_SZ, N_HEADS, device)
nn

In [ ]:
ctx = pt.stack([stot("_" * CTX_SZ).argmax(-1).to(device)])
ctx

In [ ]:
nn(ctx).shape

In [ ]:
X_batch, y_batch = X_train.argmax(-1), y_train.argmax(-1)
import copy
model = copy.deepcopy(nn)
optim = pt.optim.AdamW(model.parameters())
optim

In [ ]:
def generate(model, seed = 42, device = device, ctx_sz = 4, num_samples = 10, max_tokens = 16):
  pt.manual_seed(seed)
  samples = []
  ctx = pt.stack([stot(ctx_sz * "_").argmax(-1).to(device)])
  for _ in range(num_samples):
    sample = ctx.clone()
    for _ in range(max_tokens):
      logits = model(sample[:, -ctx_sz:])
      probs = pt.nn.functional.softmax(logits, -1)
      tok = pt.multinomial(probs, 1)
      if tok.item() == 1:
        break
      sample = pt.cat([sample, tok], 1)

    sample = dec(sample.squeeze().tolist()[ctx_sz:])
    samples.append(sample)
  return samples

## Supervised pre-training

Train the base character-level model using cross-entropy loss. This gives us a shared starting point for the RL fine-tuning that follows.

In [ ]:
EPOCHS = 2000
for epoch in range(EPOCHS):
  logits = model(X_batch)
  loss = pt.nn.functional.cross_entropy(logits, y_batch)
  if epoch % 100 == 0:
    with pt.no_grad():
      val_loss = pt.nn.functional.cross_entropy(model(X_val.argmax(-1)), y_val.argmax(-1))
      print(f"epoch {epoch:2d} train loss {loss.item():.4f} val loss {val_loss.item():.4f}")
      samples = generate(model, seed = epoch, device = device)
      print(",".join(samples))
      print()
  loss.backward()
  optim.step()
  optim.zero_grad()

# Human preferences for reinforcement learning with human feedback

In [ ]:
import re
liked_names = [name for name in src.splitlines() if bool(re.match('^[^aeiou][aeiou][^aeiou]$', name))]
liked_names[:5]

In [ ]:
names = [CTX_SZ * "_" + name + "." for name in liked_names]
names[:5]

In [ ]:
obs = pt.cat([name_to_obs(name).argmax(-1) for name in names])
obs = obs.to(device)
obs.shape

In [ ]:
actions = pt.cat([name_to_action(name).argmax(-1) for name in names])
actions = actions.to(device)
actions.shape

In [ ]:
def reward(name, gamma = .9):
  idx = pt.arange(len(name) - 1, -1, -1).to(device)
  return gamma ** idx

reward(names[0][CTX_SZ:])

In [ ]:
rewards = pt.cat([reward(name[CTX_SZ:]) for name in names])
rewards[:10], actions[:10]

## MaxRL: Maximum Likelihood Reinforcement Learning

MaxRL differs from REINFORCE by a single, critical change in how the policy gradient is normalized:

| | REINFORCE | MaxRL |
|---|---|---|
| **Gradient** | $\frac{1}{N}\sum_{i=1}^{N} r_i \nabla_\theta \log \pi_\theta(z_i \mid x)$ | $\frac{1}{K}\sum_{i=1}^{N} r_i \nabla_\theta \log \pi_\theta(z_i \mid x)$ |
| **Normalization** | Total samples $N$ | Successful samples $K = \sum_i r_i$ |

**Why does this matter?** Normalizing by $K$ (successful trajectories) instead of $N$ (all trajectories) makes the gradient equivalent to the **conditional expectation of the score function given success** (Theorem 1 in the MaxRL paper). This targets a harmonic mixture of pass@k objectives, progressively upweighting harder inputs where the model rarely succeeds.

In practice, the advantage for each token is normalized by the **per-task mean reward** $\hat{r}$ rather than being left unnormalized (REINFORCE) or normalized by standard deviation (GRPO).

In [ ]:
import copy
maxrl_model = copy.deepcopy(model)
maxrl_model.train(False)
maxrl_optim = pt.optim.AdamW(maxrl_model.parameters())
maxrl_optim.zero_grad()

### MaxRL training loop

The key difference from the REINFORCE cell in `nanoRLHF.ipynb` is in the loss computation:

```python
# REINFORCE:  loss = -(rewards * log_probs).mean()        # divides by N
# MaxRL:      loss = -(rewards * log_probs).sum() / K     # divides by K (successful samples)
```

Since our reward function assigns non-zero rewards to all tokens in liked names (they are all "successful" trajectories from the preference dataset), $K$ equals the number of tokens with reward $> 0$. To make the comparison sharper, we also apply the **control variate** from the MaxRL paper: subtracting the unconditional mean score $\frac{1}{N}\sum_i S_i$ (which has zero expectation) to reduce variance.

In [ ]:
STEPS = 25
for step in range(STEPS):
  logits = maxrl_model(obs)
  log_prob_dist = pt.nn.functional.log_softmax(logits, -1)
  log_probs = log_prob_dist[pt.arange(len(actions)), actions]

  # === MaxRL gradient ===
  # Binary success mask: tokens with reward > 0 are "successful"
  success_mask = (rewards > 0).float()
  K = success_mask.sum().clamp(min=1)  # number of successful samples

  # Control variate: subtract mean log-prob (zero-expectation baseline)
  baseline = log_probs.mean()

  # MaxRL loss: normalize by K (successful samples) instead of N (total samples)
  loss = -(rewards * (log_probs - baseline)).sum() / K

  with pt.no_grad():
    # For comparison: the equivalent REINFORCE loss
    reinforce_loss = -(rewards * log_probs).mean()

    names = generate(maxrl_model, device = device, seed = 42 + step)
    new_names = set(set(names) - set(liked_names))
    print(f"step = {step:2d} maxrl_loss={loss.item():.4f} reinforce_equiv={reinforce_loss.item():.4f} net_new_names={len(new_names):2d}")
    print(",".join(new_names))

  loss.backward()
  maxrl_optim.step()
  maxrl_optim.zero_grad()

## MaxRL with PPO-style clipping

We can also combine the MaxRL normalization-by-$K$ idea with PPO's clipped surrogate objective. This gives us the trust-region stability of PPO while using MaxRL's per-task normalization. The ratio is clipped the same way as in PPO, but the advantage is divided by $K$ instead of $N$.

In [ ]:
maxrl_ppo_model = copy.deepcopy(model)
maxrl_ppo_model.train(False)

ref_model = copy.deepcopy(model)
maxrl_ppo_optim = pt.optim.Adam(maxrl_ppo_model.parameters())
maxrl_ppo_optim.zero_grad()

In [ ]:
STEPS = 25
for step in range(STEPS):
  logits = maxrl_ppo_model(obs)
  log_probs = pt.nn.functional.log_softmax(logits, -1)[pt.arange(len(actions)), actions]
  with pt.no_grad():
    ref_log_probs = pt.nn.functional.log_softmax(ref_model(obs), -1)[pt.arange(len(actions)), actions]

  ratio = (log_probs - ref_log_probs).exp()

  # MaxRL normalization: divide by K (successful samples) instead of N
  success_mask = (rewards > 0).float()
  K = success_mask.sum().clamp(min=1)

  ppo_loss1 = rewards * ratio
  ppo_loss2 = rewards * pt.clamp(ratio, .8, 1.2)

  # MaxRL+PPO: use sum/K instead of mean (sum/N)
  loss = -pt.min(ppo_loss1, ppo_loss2).sum() / K

  log_lik_loss = -(rewards * log_probs).mean()

  with pt.no_grad():
    names = generate(maxrl_ppo_model, device = device, seed = 42 + step)
    new_names = set(set(names) - set(liked_names))
    print(f"step = {step:2d} maxrl_ppo_loss={loss.item():.4f} log likelihood {log_lik_loss:.4f} net_new_names={len(new_names):2d}")
    print(",".join(new_names))

  loss.backward()
  maxrl_ppo_optim.step()
  maxrl_ppo_optim.zero_grad()

## Summary

| Algorithm | Key change | What it optimizes |
|---|---|---|
| **REINFORCE** | $\frac{1}{N}\sum r_i S_i$ | pass@1 |
| **MaxRL** | $\frac{1}{K}\sum r_i S_i$ | harmonic mixture of pass@k |
| **PPO** | Clipped ratio, $\frac{1}{N}$ | pass@1 with trust region |
| **MaxRL+PPO** | Clipped ratio, $\frac{1}{K}$ | pass@k mixture with trust region |

The single-line change from $N$ to $K$ normalization shifts the objective from optimizing only first-attempt success (pass@1) to a broader objective that also improves the model's ability to succeed given multiple attempts (pass@k). This is particularly useful for harder inputs where the model's success rate is low.